# Fongbe Challenge — Pipeline Colab v2

1. CSV/scripts depuis **Drive partagé**, tars audio depuis le **bucket GCS public** → extraction → suppression des tars.
2. Drive monté **uniquement** pour les checkpoints partagés + modèle final.
3. `train_w2vbert.py` : checkpoint auto / 500 steps, reprise auto multi-comptes.

Runtime : **GPU T4**.

## 0. Settings

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/fongbe')
DATA_DIR.mkdir(parents=True, exist_ok=True)

REPO_RAW = 'https://raw.githubusercontent.com/Appolinairee/fongbe-challenge/master'
GCS_BASE = 'https://storage.googleapis.com/fongbe'
TAR_PREFIXES = ['audio_train_reliable_']
DRIVE_OUT = Path('/content/drive/MyDrive/fongbe-challenge/outputs/w2vbert_shared')

print('settings ok')

## 1. CSV + scripts depuis Drive (partagé)

In [ ]:
# Monte Drive et copie les petits fichiers (CSV + scripts, ~20 MB)
from google.colab import drive
from pathlib import Path
import shutil
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # déjà monté

DRIVE_DATA = Path('/content/drive/MyDrive/fongbe-challenge/data')
for f in DRIVE_DATA.iterdir():
    dst = Path('/content/fongbe') / f.name
    if f.suffix in ('.csv', '.py', '.md') and not dst.exists():
        shutil.copy2(f, dst)
        print('✓', f.name)
print('data files ok:', len(list(Path('/content/fongbe').glob('*.csv'))), 'CSV')

## 2. Télécharger + extraire les archives audio

Tar → extrait dans `DATA_DIR` → **supprimé** (économie disque). ~13 GB WAV pour reliable.

In [ ]:
import subprocess, shutil, json, urllib.request

TAR_DIR = DATA_DIR / 'tars'
TAR_DIR.mkdir(exist_ok=True)

listing = json.loads(urllib.request.urlopen(
    'https://storage.googleapis.com/storage/v1/b/fongbe/o?maxResults=200&fields=items(name,size)').read())
all_tars = sorted(i['name'] for i in listing['items'] if i['name'].endswith('.tar'))
tars = [t for t in all_tars if any(t.startswith(p) for p in TAR_PREFIXES)]
print(f'{len(tars)} archives sélectionnées')

for name in tars:
    tar_path = TAR_DIR / name
    if not tar_path.exists():
        print(f'download {name}', flush=True)
        subprocess.run(['wget','-q','-O',str(tar_path),f'{GCS_BASE}/{name}'], check=True)
    print(f'extract {name}', flush=True)
    subprocess.run(['tar','-xf',str(tar_path),'-C',str(DATA_DIR)], check=True)
    tar_path.unlink()
    print(f'ok {name}', flush=True)

free = shutil.disk_usage('/content').free/1e9
print(f'audio extrait. Disque libre : {free:.1f} GB')

In [ ]:
# Vérification WAV vs CSV
import pandas as pd
sub = DATA_DIR / 'audio/train_reliable'
n = len(list(sub.glob('*.wav'))) if sub.exists() else 0
print(f'reliable: {n:,} wav')
df = pd.read_csv(DATA_DIR/'Train_reliable.csv')
missing = (~df.audio_path.map(lambda p: (DATA_DIR/p).exists())).sum()
print(f'{len(df):,} lignes CSV, {missing} wav manquants')

## 3. Monter Drive (checkpoints + sauvegarde)

In [ ]:
from google.colab import drive
from pathlib import Path
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # déjà monté
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('drive ok →', DRIVE_OUT)

## 4. Entraînement

In [ ]:
# Dépendances + script d'entraînement depuis GitHub
import subprocess, sys, urllib.request
subprocess.run([sys.executable,'-m','pip','install','-q',
                'transformers==5.14.1','soundfile','rapidfuzz==3.14.5','bitsandbytes'], check=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/Appolinairee/fongbe-challenge/master/scripts/train_w2vbert.py',
                           '/content/train_w2vbert.py')
print('ok')

In [ ]:
# TEST SYNC — vérifie rclone→Drive 400 GB AVANT le training
import subprocess, os
from pathlib import Path
from google.colab import userdata
conf = userdata.get('RCLONE_CONF')
cfg = Path.home()/'.config/rclone'; cfg.mkdir(parents=True, exist_ok=True)
(cfg/'rclone.conf').write_text(conf)
subprocess.run(['apt-get','install','-y','-qq','rclone'], check=False)
R = 'fonvoice-drive:fongbe-challenge/outputs/w2vbert_shared'
print(subprocess.run(['rclone','lsd',R],capture_output=True,text=True).stdout)
Path('/tmp/test_sync.txt').write_text('ok')
r = subprocess.run(['rclone','copyto','/tmp/test_sync.txt',f'{R}/test_sync.txt'],capture_output=True,text=True)
print('upload rc:', r.returncode, r.stderr[-200:])
r2 = subprocess.run(['rclone','lsf',f'{R}/test_sync.txt'],capture_output=True,text=True)
print('lu distant:', r2.stdout.strip())

In [ ]:
# Entraînement — checkpoints locaux + sync Drive partagée
import subprocess, sys, os

# Secret Colab optionnel : HF_REPO (icône clé) — sinon Drive seul
RCLONE_CONF = ''
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
    HF_REPO = userdata.get('HF_REPO') or ''
    RCLONE_CONF = userdata.get('RCLONE_CONF') or ''
except Exception:
    HF_REPO = os.environ.get('HF_REPO', '')

# rclone configuré depuis le secret → écriture en tant que PROPRIÉTAIRE (quota 400 GB)
RCLONE_REMOTE = ''
if RCLONE_CONF:
    import subprocess as _sp
    from pathlib import Path as _P
    _sp.run(['apt-get','install','-y','-qq','rclone'], check=False)
    _cfg = _P.home()/'.config/rclone'; _cfg.mkdir(parents=True, exist_ok=True)
    (_cfg/'rclone.conf').write_text(RCLONE_CONF)
    RCLONE_REMOTE = 'fonvoice-drive:fongbe-challenge/outputs/w2vbert_shared'
    print('rclone → compte propriétaire (quota 400 GB)')

cmd = [
    sys.executable, '/content/train_w2vbert.py',
    '--data-dir', '/content/fongbe',
    '--work-dir', '/content/fongbe_run',
    '--max-steps', '4000',
    '--keep-checkpoints', '5',
    '--checkpoint-every', '50',
    '--batch-size', '2', '--grad-accum', '8',
]
if RCLONE_REMOTE:
    cmd += ['--sync-remote', RCLONE_REMOTE]
else:
    # sans rclone : fallback Drive FUSE (quota du compte Colab — à éviter)
    cmd += ['--sync-dir', '/content/drive/MyDrive/fongbe-challenge/outputs/w2vbert_shared']
if HF_REPO:
    cmd += ['--hf-repo', HF_REPO]
# stream en direct + log persistant /content/train.log (lisible même si crash)
log = open('/content/train.log', 'w', buffering=1)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
    log.write(line)
proc.wait()
log.close()
print('exit code:', proc.returncode)
if proc.returncode != 0:
    print('--- 30 dernières lignes du log ---')
    print(open('/content/train.log').read()[-4000:])